# Datenvorbereitung mit Python: Vom Rohdatensatz zum trainierbaren Modell

**Probe-Lehrveranstaltung – Hochschule Emden/Leer**  
Bachelor Informatik, 3. Fachsemester

Dieses Notebook begleitet die Vorlesung Schritt für Schritt anhand des **California-Housing-Datensatzes**.

## Lernziele

Nach der Veranstaltung können Sie:

- einen Rohdatensatz mit `pandas` strukturell untersuchen,
- typische Datenqualitätsprobleme erkennen,
- fehlende Werte mit unterschiedlichen Strategien behandeln,
- kategoriale Merkmale für ML-Modelle kodieren,
- numerische Merkmale sinnvoll skalieren,
- Data Leakage beim Train/Test-Split vermeiden,
- Datenvorbereitung reproduzierbar in einer `scikit-learn`-Pipeline abbilden.

## Aufbau (wie in den Folien)

0. Versuch 1: sofort trainieren
1. **Erkunden** – 1.1 pandas · 1.2 Kontext
2. **Reinigen** – 2.1 Unerwartete Werte · 2.2 Uneinheitliche Kategorien · 2.3 Unerwünschte Zeilen und Spalten
3. **Umwandeln** – erst teilen, dann lernen · 3.1 Kategoriale Werte · 3.2 Fehlende Daten · 3.3 Normalisierung
4. **Pipeline** – Versuch 3
5. **Fazit und Transfer**
- Anhang

Jede Code-Zelle, die auf einer Folie steht, finden Sie hier wortgleich wieder.

### Wichtige Bibliotheken

- **pandas**: Tabellen einlesen, untersuchen und bereinigen.
- **numpy**: numerische Operationen.
- **matplotlib**: Grafiken.
- **scikit-learn**: Datenvorbereitung (Imputer, Encoder, Scaler), Pipelines und Modelle.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)

# Einheitlicher Stil für alle Grafiken
plt.rcParams.update({
    "figure.dpi": 150, "font.size": 11, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titleweight": "bold",
})
BAR, ACCENT, GREY = "#387A87", "#009EE0", "#4B4B4D"
plt.rcParams["figure.figsize"] = (12, 8)

### Datensatz

Notebook und Datensatz liegen im Repository [github.com/larbiab/DatenVorbereitung](https://github.com/larbiab/DatenVorbereitung).
Wir laden die CSV-Datei direkt von dort – das funktioniert lokal und in Google Colab.

In [ ]:
data_url = ("https://raw.githubusercontent.com/larbiab/"
            "DatenVorbereitung/main/california_housing_dirty.csv")

## Ihr Auftrag: Hauspreise besser einschätzen

Ihre Vorgesetzte gibt Ihnen einen Datensatz. Ziel: Hauspreise in Kalifornien schätzen.
**Zielvariable im Datensatz:** `median_house_value`.

### Versuch 1: sofort trainieren

Wir laden die Datei unverändert und trainieren direkt ein Modell.
Die Zelle **endet absichtlich mit einem Fehler** (`ValueError`).

> **Was ist das Problem?**

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
housing = pd.read_csv(data_url)

X_raw = housing.drop(columns="median_house_value")
y_raw = housing["median_house_value"]

LinearRegression().fit(X_raw, y_raw)

Das Modell rechnet nur mit Zahlen. Schon die erste Spalte (`district_id`, z. B. `CA-07626`) ist Text.
Bevor wir trainieren können, müssen wir die Daten **erkunden**, **reinigen** und **umwandeln**.

# 1 Daten erkunden

Was steht in der Datei – und was würde ein Modell daraus machen?

## 1.1 pandas

Ein `DataFrame` ist eine Tabelle: Zeile = Gebiet, Spalte = Merkmal.
`pd.read_csv()` liest die CSV-Datei, `head()`, `hist()`, `info()` und `describe()` zeigen, was in den Daten steht.

### Datensatz als CSV

In [ ]:
import pandas as pd
housing = pd.read_csv(data_url)

housing.head(5)

### Verteilungen auf einen Blick

`hist()` zeichnet nur Spalten, die pandas als Zahlen erkannt hat.

> **Frage:** Was sieht ungewöhnlich aus? Welche Spalte fehlt?

In [ ]:
housing.hist(bins=50)
plt.tight_layout()
plt.show()

### Datentypen und Lücken: `info()`

> **Frage:** Welche Spalten sind verdächtig?

In [ ]:
housing.info()

### Kennzahlen: `describe()`

`describe()` berücksichtigt nur Zahlenspalten. Es fehlt genau die Zielvariable `median_house_value`.

In [ ]:
housing.describe().T

## 1.2 Kontext

- Volkszählung Kalifornien, 1990 (Pace & Barry, 1997; Bezug über StatLib).
- Eine Zeile = ein Wohngebiet (Block Group, 600–3.000 Einwohner).
- Merkmale: Lage, Räume, Alter, Bevölkerung, Einkommen, Nähe zum Meer.
- Wir schätzen den mittleren Hauswert pro Gebiet.

**Was weiß ein Modell über unsere Welt?** Ein untrainiertes Modell weiß nichts.
Alles Wissen vermitteln ihm die Daten. Datenvorbereitung übersetzt unser Weltwissen in Daten:
Wir geben dem Modell Informationen und beseitigen Widersprüche.

# 2 Daten reinigen

Fehler und Widersprüche finden, belegen und beheben. Wir halten die Bereinigung bewusst einfach – **kein Regex nötig**.

## 2.1 Unerwartete Werte

### Was ist hier das Problem?

Wir versuchen, die Preisspalte in Zahlen umzuwandeln. Was dabei zu `NaN` wird, obwohl vorher ein Wert da war, ist nicht lesbar.

In [ ]:
converted_price = pd.to_numeric(
    housing["median_house_value"], errors="coerce")
invalid_price = (converted_price.isna()
    & housing["median_house_value"].notna())

print("Nicht lesbare Preiswerte:", invalid_price.sum())
housing.loc[invalid_price, ["median_house_value", "ocean_proximity"]].head(6)

**Kleines Beispiel:** So arbeitet `pd.to_numeric(liste, errors="coerce")`.

In [ ]:
liste = pd.Series(["2983", "kasdd", "33,5", "45"])
converted = pd.to_numeric(liste, errors="coerce")
invalid = converted.isna() & liste.notna()

display(pd.DataFrame({"liste": liste, "converted": converted, "invalid": invalid}))
liste[invalid]

### Dollarzeichen im Preis entfernen

Wir arbeiten ab hier auf einer Kopie `housing_cleaned` – die Rohdaten bleiben unverändert.

In [ ]:
housing_cleaned = housing.copy()
price = housing_cleaned["median_house_value"]
price = price.str.rstrip("$")
housing_cleaned["median_house_value"] = pd.to_numeric(price)

housing_cleaned.loc[[28, 51, 73, 209, 388, 394], ["median_house_value", "ocean_proximity"]]

### Kontrolle: `info()` nach der Bereinigung

In [ ]:
housing_cleaned.info()

## 2.2 Uneinheitliche Kategorien

### Rohdaten ansehen

In [ ]:
columns_category = ["longitude", "latitude", "median_house_value", "ocean_proximity"]
rows_category = [575, 612, 1289, 1410, 2449, 3063]
housing.loc[rows_category, columns_category]

### Problem markieren: Wie viele Schreibweisen gibt es?

In [ ]:
(housing["ocean_proximity"]
    .value_counts(dropna=False))

`value_counts()` zeigt Varianten durch Groß-/Kleinschreibung und Leerzeichen.
Der Wert `near` ist **mehrdeutig** (Bay oder Ozean?) – das darf Python nicht allein entscheiden.
Wir markieren ihn deshalb als fehlend (`NaN`): Er fehlt bewusst in `category_map`.

### Bereinigt prüfen

In [ ]:
category_map = {
    "NEAR BAY": "NEAR BAY",
    "NEAR OCEAN": "NEAR OCEAN",
    "<1H OCEAN": "<1H OCEAN",
    "< 1H OCEAN": "<1H OCEAN",
    "1H OCEAN": "<1H OCEAN",
    "INLAND": "INLAND",
    "IN LAND": "INLAND",
    "ISLAND": "ISLAND",
    # "NEAR" fehlt bewusst: mehrdeutig -> NaN
}

ocean_simple = (housing_cleaned["ocean_proximity"]
    .astype("string").str.strip().str.upper())
housing_cleaned["ocean_proximity"] = (
    ocean_simple.map(category_map))

housing_cleaned.loc[rows_category, columns_category]

In [ ]:
housing_cleaned["ocean_proximity"].value_counts(dropna=False)

## 2.3 Unerwünschte Zeilen und Spalten

### Doppelte Zeilen: Rohdaten ansehen

In [ ]:
columns_duplicates = ["district_id", "longitude", "latitude", "total_rooms",
                      "median_house_value", "ocean_proximity"]
housing.loc[[2245, 14983, 6011, 1858],
            columns_duplicates]

### Problem markieren

In [ ]:
print("Zusätzliche Duplikate:",
      housing.duplicated().sum())
(housing[housing.duplicated(keep=False)]
    .sort_values("district_id")
    [columns_duplicates].head(4))

### Bereinigt prüfen: Kopien entfernen

In [ ]:
housing_cleaned = housing_cleaned.drop_duplicates()

print("Zeilen:", len(housing_cleaned))
housing_cleaned.loc[[1858, 2245], columns_duplicates]

### ID-Spalte entfernen

`district_id` nummeriert nur durch – sie beschreibt das Gebiet nicht.

In [ ]:
housing_cleaned = housing_cleaned.drop(
    columns="district_id")

housing_cleaned.loc[[1858, 2245], columns_duplicates[1:]]

### Kurzer Qualitätscheck nach dem Reinigen

In [ ]:
print("Zeilen, Spalten:", housing_cleaned.shape)
print("Duplikate:", housing_cleaned.duplicated().sum())
print("ID vorhanden:", "district_id" in housing_cleaned.columns)
print("Kategorien:", sorted(housing_cleaned["ocean_proximity"].dropna().unique()))
print("Fehlende Werte je Spalte:")
housing_cleaned.isna().sum()

# 3 Daten umwandeln

Bis hierhin haben wir **regelbasiert** bereinigt – mit unserem Wissen über die Welt.
Ab jetzt **lernen** wir Größen aus den Daten: die Kategorien für das Encoding, den Median für Lücken,
Mittelwert und Standardabweichung für die Skalierung.

## Erst teilen, dann lernen

> Erst Train/Test-Split, dann alles Lernende **nur auf den Trainingsdaten** fitten.

Die Testdaten spielen „neue Daten“, die das Modell nie gesehen hat. Sonst entsteht **Data Leakage**.

In [ ]:
train_set, test_set = train_test_split(
    housing_cleaned, test_size=0.2, random_state=42)

print("Training:", train_set.shape)
print("Test:", test_set.shape)

Merkmale (`X`) und Zielvariable (`y`) trennen:

In [ ]:
X_train = train_set.drop(columns="median_house_value")
y_train = train_set["median_house_value"]

X_test = test_set.drop(columns="median_house_value")
y_test = test_set["median_house_value"]

X_train.head()

## 3.1 Kategoriale Werte

`ocean_proximity` ist Text. Zahlen 1, 2, 3 würden eine Reihenfolge behaupten (`INLAND < NEAR BAY`?).
**One-Hot-Encoding** macht aus jeder Kategorie eine eigene 0/1-Spalte.

### Das Modell braucht eine Zahlendarstellung

In [ ]:
X_train[["ocean_proximity"]].head()

### One-Hot-Encoding

Der Encoder lernt die Kategorien **nur aus `X_train`**. Fünf Kategorien plus `NaN` (aus dem mehrdeutigen `near`) ergeben sechs Spalten.

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore", sparse_output=False)
ocean_encoded = encoder.fit_transform(
    X_train[["ocean_proximity"]])
pd.DataFrame(ocean_encoded[:5],
    columns=encoder.get_feature_names_out())

### One-Hot-Encoding: nur noch Zahlen

In [ ]:
X_train_encoded = pd.concat([
    X_train.drop(columns="ocean_proximity"),
    pd.DataFrame(ocean_encoded, index=X_train.index,
                 columns=encoder.get_feature_names_out())], axis=1)
X_train_encoded.info()

Weitere Verfahren: Ordinal Encoding, Frequency Encoding, Target Encoding, Embeddings.
Für wenige nominale Kategorien ist One-Hot-Encoding eine gute und transparente Wahl.

## 3.2 Fehlende Daten

Drei Möglichkeiten: betroffene Zeilen löschen (`dropna`), die Spalte löschen (`drop`) oder **ergänzen (imputieren)**.

### Fehlende Werte: Trainingsdaten ansehen

In [ ]:
columns_missing = ["total_rooms", "total_bedrooms", "population", "households"]
# vier Zeilen mit Lücke + Zeile 269
rows_missing = list(X_train.index[
    X_train["total_bedrooms"].isna()][:4]) + [269]
X_train.loc[rows_missing, columns_missing]

### Problem markieren

In [ ]:
X_train.isna().sum()

### Wo liegen die vorhandenen Werte?

Fehlende Werte erscheinen im Histogramm nicht als Balken.

In [ ]:
X_train["total_bedrooms"].hist(bins=50, figsize=(7, 4))
plt.xlabel("total_bedrooms")
plt.ylabel("Anzahl Gebiete")
plt.show()

### Ergänzen: Median der Trainingsdaten

Die Verteilung ist rechtsschief: Wenige sehr große Gebiete ziehen den Mittelwert hoch – der Median bleibt beim typischen Gebiet.

In [ ]:
num_preview = X_train.select_dtypes(include=np.number)
imputer = SimpleImputer(strategy="median")
imputed = imputer.fit_transform(num_preview)

imputed_df = pd.DataFrame(imputed, columns=num_preview.columns,
                          index=num_preview.index)
print("Median total_bedrooms (Training):", imputed_df.loc[rows_missing[0], "total_bedrooms"])
print("Mittelwert total_bedrooms (Training):", round(X_train["total_bedrooms"].mean(), 1))
imputed_df.loc[rows_missing, columns_missing]

### Ergänzt – aber prüfen!

Ist die Regel *Schlafzimmer ≤ Räume* nach der Imputation noch erfüllt?

In [ ]:
rule_after = imputed_df["total_bedrooms"] > imputed_df["total_rooms"]
print("Regelverstöße nach Median-Imputation:", rule_after.sum())
imputed_df.loc[rule_after, columns_missing]

### Median ergänzen und sichtbar prüfen

In [ ]:
bedrooms_before = X_train["total_bedrooms"].astype(float)
bedrooms_after = imputed_df["total_bedrooms"]
median_value = imputer.statistics_[list(num_preview.columns).index("total_bedrooms")]
print(f"Fehlend im Training: {bedrooms_before.isna().sum()} | Median: {median_value:.0f}")

bins = np.arange(0, 1850, 100)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
axes[0].hist(bedrooms_before.dropna().clip(upper=1800), bins=bins, color=BAR, edgecolor="white")
axes[0].set_title("Vorher · nur beobachtete Werte")
axes[1].hist(bedrooms_after.clip(upper=1800), bins=bins, color=BAR, edgecolor="white")
axes[1].hist(np.full(bedrooms_before.isna().sum(), median_value), bins=bins, color=ACCENT,
             edgecolor="white", label=f"ergänzt: {bedrooms_before.isna().sum()} × Median {median_value:.0f}")
axes[1].set_title("Nachher · Median ergänzt")
axes[1].legend(frameon=False)
for ax in axes:
    ax.set_xlabel("total_bedrooms (Werte über 1.800 zusammengefasst)")
axes[0].set_ylabel("Anzahl Gebiete")
plt.tight_layout()
plt.show()

### Ergänzt: keine Lücken mehr

In [ ]:
X_train_num = pd.DataFrame(imputed,
    columns=num_preview.columns)
X_train_num.info()

Weitere Strategien: Mittelwert, häufigster Wert, konstante Ersatzwerte, KNN-Imputation, modellbasierte Imputation.
Es gibt **nicht die eine richtige Strategie** – ergänzte Werte bleiben Schätzungen.

## 3.3 Normalisierung

`median_income` zählt in 10.000 $, `total_rooms` in Räumen – ein Unterschied von 1 bedeutet etwas völlig anderes.

### Unterschiedliche Skalen

In [ ]:
(X_train[["median_income", "population",
          "total_rooms"]]
    .describe().T[["min", "max", "mean", "std"]])

### Standardisieren mit `StandardScaler`

$z = \frac{x - \mu}{\sigma}$ – $\mu$ und $\sigma$ werden **nur aus dem Training** gelernt (`fit`) und auf Test und neue Daten angewendet (`transform`).

In [ ]:
scaler = StandardScaler()
scaled = scaler.fit_transform(
    X_train[["median_income", "population", "total_rooms"]])
scaled_test = scaler.transform(
    X_test[["median_income", "population", "total_rooms"]])

pd.DataFrame(scaled, columns=["median_income", "population", "total_rooms"],
             index=X_train.index).head().round(2)

In [ ]:
print("Mittelwerte (Training):", [float(round(v, 2)) for v in scaler.mean_])
print("Standardabweichungen (Training):", [float(round(v, 2)) for v in scaler.scale_])

pd.DataFrame(scaled_test, columns=["median_income", "population", "total_rooms"]).describe().T[["mean", "std"]].round(2)

### Alternative Skalierungsstrategien

- `StandardScaler` → Mittelwert 0, Standardabweichung 1
- `MinMaxScaler` → typischerweise Werte zwischen 0 und 1
- `RobustScaler` → robuster gegenüber Ausreißern
- keine Skalierung

Wichtig:

> **Nicht jedes Modell benötigt Skalierung.**

Beispiele:

- k-NN, SVM, neuronale Netze: oft wichtig
- Entscheidungsbäume, Random Forests: häufig weniger relevant

# 4 Pipeline

Bis jetzt haben wir jeden Schritt von Hand ausgeführt. Ein neuer Datensatz muss **exakt dieselben Transformationen** erhalten.
Dafür bündeln wir alles in einer **Pipeline**:

1. Zeichen und Kategorien bereinigen – im Notebook: Abschnitt 2 (Reinigen)
2. `ColumnTransformer` – Zahlen: `SimpleImputer` → `StandardScaler`; Text: `OneHotEncoder`
3. Modell

`fit` nur auf den Trainingsdaten, `transform` auf Testdaten und neue Daten.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns
categorical_features = X_train.select_dtypes(exclude=np.number).columns

numeric_features, categorical_features

In [ ]:
numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

preprocessing = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessing

In [ ]:
X_train_prepared = preprocessing.fit_transform(X_train)
X_test_prepared = preprocessing.transform(X_test)

print("Trainierbare Matrix:", X_train_prepared.shape)
print("Testmatrix:", X_test_prepared.shape)

### Versuch 3: Das Modell lernt

Jetzt bekommt das Modell Daten, in die wir unser Wissen über die Welt übersetzt haben.

In [ ]:
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", LinearRegression())])
full_pipeline.fit(X_train, y_train)
full_pipeline.predict(X_test.head())

In [ ]:
pd.DataFrame({
    "Vorhersage": full_pipeline.predict(X_test.head()).round(2),
    "Tatsächlicher Wert": y_test.head().to_numpy()
})

# 5 Fazit und Transfer

Vom Rohdatensatz zum trainierbaren Modell:

1. **Erkunden** – pandas (`head`, `hist`, `info`, `describe`) und Kontext
2. **Reinigen** – `$` entfernen, Schreibweisen vereinheitlichen, Duplikate und ID entfernen
3. **Umwandeln** – erst teilen, dann lernen: One-Hot-Encoding, Median-Imputation, Standardisierung (alles `fit` nur auf `X_train`)
4. **Pipeline** – alles reproduzierbar in einem Ablauf

### Wann sind Daten bereit?

1. Verstanden · 2. Geprüft · 3. Begründet bereinigt · 4. Reproduzierbar transformiert

**Datenvorbereitung übersetzt Weltwissen in Daten: Informationen geben, Widersprüche beseitigen.**

## Transferfrage

Angenommen, wir hätten statt Immobilienpreisen:

- Sensordaten eines Roboters,
- medizinische Zeitreihen,
- Bilder,
- Nutzungsdaten einer App.

> **Welche Schritte unserer Pipeline könnten wir beibehalten – und welche müssten wir ändern?**

# Anhang

Zusatzauswertungen für Rückfragen (Folien im Anhang).

### `describe()` für die Preisspalte (Text)

In [ ]:
housing[["median_house_value"]].describe().T

### Verteilungen nach der Bereinigung

In [ ]:
housing_cleaned.hist(bins=50)
plt.tight_layout()
plt.show()

### Grenzwerte prüfen und dokumentieren

In [ ]:
df = housing_cleaned
print((df['housing_median_age'] == 52).sum())
print((df['median_house_value']
    == 500001).sum())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 6.2))
for ax, column, cap, label in [
    (axes[0], "housing_median_age", 52, "52 Jahren"),
    (axes[1], "median_house_value", 500001, "500.001 $"),
]:
    values = df[column].astype(float)
    counts, edges, patches = ax.hist(values, bins=50, color=BAR, edgecolor="white")
    patches[-1].set_facecolor(ACCENT)
    n_cap = (values == cap).sum()
    ax.set_title(column)
    ax.text(0.02, 0.95, f"{n_cap:,} Gebiete genau bei {label}".replace(",", "."),
            transform=ax.transAxes, va="top", color=ACCENT, fontweight="bold")
    ax.set_ylabel("Anzahl Gebiete")
plt.tight_layout()
plt.show()

### Was hängt mit dem Preis zusammen?

In [ ]:
numeric_columns = housing_cleaned.select_dtypes(include=np.number).columns
correlations = (
    df[numeric_columns].astype(float).corr()["median_house_value"]
    .drop("median_house_value").sort_values()
)
display(correlations.round(2).to_frame("Korrelation mit median_house_value"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
colors = [ACCENT if name == "median_income" else BAR for name in correlations.index]
axes[0].barh(correlations.index, correlations.values, color=colors)
axes[0].axvline(0, color=GREY, linewidth=0.8)
axes[0].set_title("Korrelation mit median_house_value")
axes[1].scatter(df["median_income"], df["median_house_value"], s=4, alpha=0.15, color=BAR)
axes[1].set_xlabel("median_income (in 10.000 $)")
axes[1].set_ylabel("median_house_value ($)")
axes[1].set_title("Einkommen und Hauswert")
plt.tight_layout()
plt.show()

### Wo liegen die teuren Gebiete?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
points = ax.scatter(df["longitude"], df["latitude"],
    c=df["median_house_value"], cmap="viridis",
    s=df["population"] / 100, alpha=0.4)
fig.colorbar(points, ax=ax, label="median_house_value ($)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
plt.show()

### Versuch 2: Reicht die Bereinigung?

Die Zelle **endet absichtlich mit einem Fehler**: Bereinigt heißt noch nicht modellbereit.

In [ ]:
X_clean = housing_cleaned.drop(
    columns="median_house_value")
y_clean = housing_cleaned["median_house_value"]
LinearRegression().fit(X_clean, y_clean)

### Ausblick: Feature Engineering

## 10. Datenvorbereitung ist mehr als Cleaning: Feature Engineering

Ein einzelner Rohwert muss nicht immer die beste Repräsentation sein.

Zum Beispiel:

```text
total_rooms = 2356
households  = 481
```

Eine möglicherweise informativere Größe ist:

\[
rooms\_per\_household =
\frac{total\_rooms}{households}
\]

In [ ]:
X_train_fe = X_train.copy()

X_train_fe["rooms_per_household"] = (
    X_train_fe["total_rooms"] / X_train_fe["households"]
)

X_train_fe[["total_rooms", "households", "rooms_per_household"]].head()

### Drei unterschiedliche Aufgaben

**Cleaning**  
Fehler und fehlende Werte behandeln.

**Transformation**  
Daten in eine für das Modell geeignete Darstellung bringen.

**Feature Engineering**  
Aus vorhandenen Informationen sinnvollere Merkmale erzeugen.

### Alternative Split-Strategien

Ein zufälliger Split ist nicht immer die beste Wahl.

| Situation | Geeignete Strategie |
|---|---|
| allgemeiner IID-Datensatz | Random Split |
| wichtige Klassen/Verteilungen erhalten | Stratified Split |
| Zeitreihen | zeitbasierter Split |
| mehrere Messungen derselben Person/Gruppe | Group Split |

**Merksatz:**  
Die Split-Strategie muss zur Datenentstehung passen.

## Literatur

Aurélien Géron:  
*Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*, 3rd Edition, O'Reilly, 2022.  
Insbesondere Kapitel 2: **End-to-End Machine Learning Project**.

Wes McKinney:  
*Python for Data Analysis*, 3rd Edition, O'Reilly, 2022.